# CIS6005 Computational Intelligence
## Notebook 06 — Data Preprocessing (Encoding + Scaling)
**Student Health Risk Prediction | Kaggle PS S6E7**

---
### Why Preprocessing?
ML algorithms operate on **numbers only**. Preprocessing bridges this gap:

| Problem | Solution |
|---------|----------|
| Categorical text columns | Label Encoding |
| Features on different scales | StandardScaler |
| Target class as string | LabelEncoder on target |

> **CRITICAL RULE:** All encoders/scalers must be **fit on training data only**, then **transform-only on test data**.

In [1]:
# ============================================================
# IMPORTS & SETUP
# ============================================================
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent
PROC_DATA    = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR   = PROJECT_ROOT / 'models'

train_df = pd.read_csv(PROC_DATA / 'train_featured.csv')
test_df  = pd.read_csv(PROC_DATA / 'test_featured.csv')

print(f'Loaded: Train {train_df.shape} | Test {test_df.shape}')

Loaded: Train (690088, 21) | Test (295753, 20)


## 1. Separate Features and Target

In [2]:
# ============================================================
# SECTION 1: Separate Features and Target
# ============================================================

TARGET = 'health_condition'

X = train_df.drop(columns=['id', TARGET], errors='ignore')
y = train_df[TARGET]

test_ids   = test_df['id'] if 'id' in test_df.columns else pd.Series(range(len(test_df)))
X_test_raw = test_df.drop(columns=['id', TARGET], errors='ignore')

print(f'X shape      : {X.shape}')
print(f'y shape      : {y.shape}')
print(f'X_test shape : {X_test_raw.shape}')
print(f'Classes      : {list(y.unique())}')

X shape      : (690088, 19)
y shape      : (690088,)
X_test shape : (295753, 19)
Classes      : ['unhealthy', 'at-risk', 'fit']


## 2. Encode Target Variable

**Why:** Scikit-learn classifiers require numeric integer targets.
LabelEncoder maps: `at-risk→0, fit→1, unhealthy→2`

In [3]:
# ============================================================
# SECTION 2: Encode Target Variable
# ============================================================

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

class_mapping = dict(zip(
    label_encoder.classes_,
    label_encoder.transform(label_encoder.classes_)
))

print('Target Encoding Map:')
for cls_name, code in class_mapping.items():
    print(f'  "{cls_name}" -> {code}')

joblib.dump(label_encoder, MODELS_DIR / 'label_encoder.joblib')
print('\n✅ Label encoder saved')

Target Encoding Map:
  "at-risk" -> 0
  "fit" -> 1
  "unhealthy" -> 2

✅ Label encoder saved


## 3. Encode Categorical Features

**Strategy:** LabelEncoding for tree-based models (safe and efficient).
Fit on TRAIN, transform BOTH — prevents data leakage.

In [4]:
# ============================================================
# SECTION 3: Encode Categorical Features
# ============================================================

categorical_feature_cols = X.select_dtypes(include='object').columns.tolist()
print(f'Encoding: {categorical_feature_cols}')

categorical_encoders = {}

for col in categorical_feature_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    
    # Handle unseen categories in test
    test_col = X_test_raw[col].astype(str)
    known    = set(le.classes_)
    test_col = test_col.apply(lambda v: v if v in known else le.classes_[0])
    X_test_raw[col] = le.transform(test_col)
    
    categorical_encoders[col] = le
    print(f'  {col}: {list(le.classes_)} -> {list(le.transform(le.classes_))}')

joblib.dump(categorical_encoders, MODELS_DIR / 'categorical_encoders.joblib')
print('\n✅ Categorical encoders saved')

Encoding: ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender', 'bmi_category']


  diet_type: ['balanced', 'non-veg', 'veg'] -> [np.int64(0), np.int64(1), np.int64(2)]


  stress_level: ['high', 'low', 'medium'] -> [np.int64(0), np.int64(1), np.int64(2)]


  sleep_quality: ['average', 'good', 'poor'] -> [np.int64(0), np.int64(1), np.int64(2)]


  physical_activity_level: ['active', 'moderate', 'sedentary'] -> [np.int64(0), np.int64(1), np.int64(2)]


  smoking_alcohol: ['no', 'occasional', 'yes'] -> [np.int64(0), np.int64(1), np.int64(2)]


  gender: ['female', 'male', 'other'] -> [np.int64(0), np.int64(1), np.int64(2)]


  bmi_category: ['normal', 'obese', 'overweight', 'underweight'] -> [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]

✅ Categorical encoders saved


## 4. Feature Scaling

**Why:** Distance-based models (KNN, SVM, Logistic Regression) are severely affected by feature scale differences.
StandardScaler: `z = (x - mean) / std` → mean≈0, std≈1 per feature.

In [5]:
# ============================================================
# SECTION 4: StandardScaler — Fit on TRAIN, Transform BOTH
# ============================================================

scaler = StandardScaler()

X          = X.apply(pd.to_numeric, errors='coerce').fillna(0)
X_test_raw = X_test_raw.apply(pd.to_numeric, errors='coerce').fillna(0)
X_test_raw = X_test_raw.reindex(columns=X.columns, fill_value=0)

X_scaled      = pd.DataFrame(scaler.fit_transform(X),      columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_raw), columns=X.columns)

print(f'X_scaled shape      : {X_scaled.shape}')
print(f'X_test_scaled shape : {X_test_scaled.shape}')

joblib.dump(scaler, MODELS_DIR / 'scaler.joblib')
print('✅ Scaler saved')

X_scaled shape      : (690088, 19)
X_test_scaled shape : (295753, 19)
✅ Scaler saved


## 5. Train / Validation Split (80% / 20%)

**Why stratify?** Ensures each class is equally proportioned in both splits. Critical for imbalanced data.

In [6]:
# ============================================================
# SECTION 5: Stratified Train / Validation Split
# ============================================================

RANDOM_STATE = 42

X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y_encoded,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_encoded
)

print(f'X_train: {X_train.shape} | X_val: {X_val.shape}')

# Class distribution
unique, counts = np.unique(y_train, return_counts=True)
print('\nClass distribution in y_train:')
for cls, cnt in zip(unique, counts):
    name = label_encoder.inverse_transform([cls])[0]
    print(f'  Class {cls} ({name}): {cnt} ({cnt/len(y_train)*100:.1f}%)')

X_train: (552070, 19) | X_val: (138018, 19)

Class distribution in y_train:
  Class 0 (at-risk): 474049 (85.9%)
  Class 1 (fit): 31842 (5.8%)
  Class 2 (unhealthy): 46179 (8.4%)


## 6. Save All Preprocessed Data

In [7]:
# ============================================================
# SECTION 6: Save Preprocessed Arrays and Feature Names
# ============================================================

np.save(PROC_DATA / 'X_train.npy', X_train.values)
np.save(PROC_DATA / 'X_val.npy',   X_val.values)
np.save(PROC_DATA / 'y_train.npy', y_train)
np.save(PROC_DATA / 'y_val.npy',   y_val)
np.save(PROC_DATA / 'X_test.npy',  X_test_scaled.values)

feature_names = list(X.columns)
joblib.dump(feature_names, MODELS_DIR / 'feature_names.joblib')

print('=' * 60)
print('  PHASE 6 COMPLETE — Preprocessing')
print('=' * 60)
print('  ✅ X_train.npy | X_val.npy | y_train.npy | y_val.npy')
print('  ✅ X_test.npy')
print('  ✅ scaler.joblib | label_encoder.joblib')
print('  ✅ categorical_encoders.joblib | feature_names.joblib')
print('=' * 60)
print('  ✅ Ready for Phase 7: Model Development')
print('=' * 60)


  PHASE 6 COMPLETE — Preprocessing
  ✅ X_train.npy | X_val.npy | y_train.npy | y_val.npy
  ✅ X_test.npy
  ✅ scaler.joblib | label_encoder.joblib
  ✅ categorical_encoders.joblib | feature_names.joblib
  ✅ Ready for Phase 7: Model Development
